# IMQCAM API Access for Modelers

This notebook demonstrates how to use the IMQCAM API to find and access files programmatically.

The main workflow is:

1. Connect to IMQCAM.
2. Choose an IGSN and data type.
3. Query the `imqcam/datafiles` endpoint.
4. Retrieve metadata such as composition.
5. Review the matching files.
6. Download files when needed.

## Key concepts

**API**  
An API allows Python code to communicate directly with the IMQCAM data portal without manually browsing through folders in the web interface.

**Endpoint**  
An endpoint is a specific API location used for a particular type of request. In this notebook, we use:

`imqcam/datafiles`

This endpoint is used to retrieve data-file records from IMQCAM.

**GET request**  
A GET request asks the API to return information. It does not modify or delete data.

**Parameters**  
Parameters tell the API what data we want. Examples include the data type, IGSN, pagination limit, and metadata fields.

**Metadata**  
Metadata describes the scientific data. Examples include IGSN, composition, file name, data type, and other descriptive information.


## Before you start

Keep this notebook in the IMQCAM project/repository where `imqcam.py` is available.

The helper file `imqcam.py` handles the connection to the IMQCAM API. If the imports below fail, confirm that `imqcam.py` is in this folder or one of its parent folders.

Create a `.env` file, add your API key, and save it.

If needed, install the supporting packages in the same Python/Jupyter environment:

```python
%pip install pandas python-dotenv girder-client
```

## 1. Connect to IMQCAM

This cell creates an authenticated IMQCAM client.

The client knows the IMQCAM API base URL and is used for all later API requests.


In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "imqcam.py").exists())
sys.path.insert(0, str(REPO_ROOT))

from imqcam import get_client, fetch_entries, entries_to_frame, extract_igsn

client = get_client()
print(f"Connected to {client.urlBase}")

## 2. Choose the specimen and data type

The two most important inputs are:

- **IGSN** — identifies the specimen or sample.
- **DATA_TYPE** — identifies the type of scientific data to retrieve, such as EBSD raw data.

Modelers normally only need to change these values for a different specimen or dataset.


In [ ]:
IGSN = "CMXMAG00001-042"
DATA_TYPE = "ebsd_raw"     

# Optional local filter after the API results are returned.
# Examples: "IN718", "Ti64"
# Use None if you do not want to filter by composition.
COMPOSITION_FILTER = None

print("IGSN:", IGSN)
print("Data type:", DATA_TYPE)
print("Composition filter:", COMPOSITION_FILTER or "None")

## 3. Query the data-files endpoint

This cell sends a GET request to:

`imqcam/datafiles`

The request asks IMQCAM for records that match the selected specimen and data type.

### What the parameters mean

- `dataType` — the type of data requested.
- `meta.igsn` — filters the results to the selected specimen.
- `extraFields` — asks the API to return additional metadata, such as composition.
- `limit` — controls how many records are returned in one request.
- `offset` — is used for pagination when there are more records than one request can return.

In plain language, the request means:

> Give me the matching IMQCAM data files for this specimen and data type, and include composition metadata.


### 4. Constituent IGSN matching

Some IMQCAM records can contain multiple comma-separated values in `meta.igsn`.

To avoid missing those records, this notebook does not rely on exact equality such as:

`meta.igsn == "CMXMAG00001-042"`

Instead, it:
1. retrieves the selected data type with pagination,
2. splits both requested and stored IGSNs into constituent values, and
3. keeps a record when at least one IGSN overlaps.

The search accepts a single IGSN, a comma-separated string, or a Python list of IGSNs.


In [ ]:
def split_igsns(value):
    """
    Normalize one or many IGSNs into a clean list.

    Supported inputs:
      - "CMXMAG00001-042"
      - "CMXMAG00001-042,CMXMAG00001-039"
      - ["CMXMAG00001-042", "CMXMAG00001-039"]

    This also handles stored IMQCAM metadata where multiple IGSNs are
    represented as a comma-separated string.
    """
    if value is None:
        return []

    if isinstance(value, (list, tuple, set)):
        result = []
        for part in value:
            result.extend(split_igsns(part))
        return result

    text = str(value).strip()

    if not text or text.lower() in {"nan", "none"}:
        return []

    return [
        part.strip()
        for part in text.split(",")
        if part.strip()
    ]


def get_all_files(
    client,
    igsn,
    data_type,
    page_size=100,
):
    """
    Retrieve all records for a data type, then match one or many requested
    constituent IGSNs locally.

    This avoids exact server-side equality on meta.igsn, which can miss
    records whose metadata contains multiple comma-separated IGSNs.
    """
    all_items = []
    offset = 0

    while True:
        batch = client.get(
            "imqcam/datafiles",
            parameters={
                "dataType": data_type,
                "extraFields": json.dumps([
                    "meta.composition"
                ]),
                "limit": page_size,
                "offset": offset,
            },
        )

        all_items.extend(batch)

        if len(batch) < page_size:
            break

        offset += page_size

    requested_igsns = set(split_igsns(igsn))

    if not requested_igsns:
        return all_items

    matching_items = []

    for item in all_items:
        stored_igsns = set(
            split_igsns(
                item.get("meta", {}).get("igsn")
            )
        )

        # Keep the record if any requested IGSN is present in its stored metadata.
        if requested_igsns & stored_igsns:
            matching_items.append(item)

    print(
        f"Scanned {len(all_items):,} {data_type} records; "
        f"matched {len(matching_items):,} record(s)."
    )

    return matching_items


 Choose the specimen and data type

The two most important inputs are:

- **IGSN** — identifies the specimen or sample.
- **DATA_TYPE** — identifies the type of scientific data to retrieve, such as EBSD raw data.

Modelers normally only need to change these values for a different specimen or dataset.


In [ ]:
items = get_all_files(
    client=client,
    igsn=IGSN,
    data_type=DATA_TYPE,
)

print(f"\nTotal matching API records: {len(items):,}")

## 5. Review metadata and composition

The actual file contains the scientific measurement data.

Metadata describes that file or specimen. In this notebook, one important metadata field is:

`meta.composition`

For example, composition metadata may identify the material associated with a specimen.

This is useful for modelers because they can confirm that the retrieved files belong to the expected material before downloading or processing them.


In [ ]:
preview_rows = []

for item in items:
    meta = item.get("meta", {}) or {}

    preview_rows.append({
        "file_name": item.get("name"),
        "igsn": meta.get("igsn"),
        "composition": meta.get("composition"),
        "item_id": item.get("_id"),
        "size": item.get("size"),
    })

files_df = pd.DataFrame(preview_rows)

print(f"Records in table: {len(files_df):,}")
display(files_df.head(5))

In [ ]:
DOWNLOAD_FILES = False
DOWNLOAD_LIMIT = 5

if DOWNLOAD_FILES:
    selected_ids = set(files_df["item_id"].dropna())

    items_to_download = [
        item for item in items
        if item.get("_id") in selected_ids
    ]

    if DOWNLOAD_LIMIT is not None:
        items_to_download = items_to_download[:DOWNLOAD_LIMIT]

    download_folder = REPO_ROOT / "downloads" / DATA_TYPE / IGSN
    download_folder.mkdir(parents=True, exist_ok=True)

    print(f"Downloading {len(items_to_download):,} item(s)...")
    print(f"Destination: {download_folder}")

    for number, item in enumerate(items_to_download, start=1):
        client.downloadItem(
            item["_id"],
            dest=str(download_folder)
        )

        print(
            f"[{number}/{len(items_to_download)}] "
            f"{item.get('name', item['_id'])}"
        )

    print("Download complete.")

else:
    print(
        "Downloads are OFF. Review the results first, then set "
        "DOWNLOAD_FILES = True when ready."
    )

## Summary for modelers

The API workflow can be remembered as:

**IGSN → Data Type → API Endpoint → Metadata → Matching Files → Download**

### Important terms

- **API** — allows Python to communicate with IMQCAM.
- **Endpoint** — the specific API resource being requested.
- **GET** — retrieves information.
- **IGSN** — identifies the specimen.
- **Data Type** — identifies the category of scientific data.
- **Metadata** — describes the data, such as composition.
- **Pagination** — retrieves large result sets in multiple batches.

You do not need to become an API developer to use this workflow. In most cases, you only need to provide the specimen IGSN and the data type, then run the notebook cells in order.
